In [35]:
'''
整体架构：
状态转移矩阵P[位置索引][动作]，其元素是一个四元组(概率，下一步位置，奖励，是否完成)
qsa_list 一个位置的Q值序列(包含4个动作)
价值函数v[x][y]二维数组，没有动作，表示每个位置的价值期望
'''
import copy

class CliffWalkingEnv:
    def __init__(self,ncol=12,nrow=4):
        self.ncol=ncol
        self.nrow=nrow
        self.P=self.createP() # P=='Path'
    
    def createP(self):
        P = [[[] for j in range(4)] for i in range(self.nrow * self.ncol)]
        #初始化一个空的集合，4*12栅格地图*4状态，每个状态元组包含4个变量(概率，下一步位置，奖励，是否完成)
        change=[[0,-1],[0,1],[-1,0],[1,0]]
         # 定义4个动作的方向向量 [Δx, Δy]
        # 上: (0, -1) → x不变，y减少1（向上移动）
        # 下: (0, 1)  → x不变，y增加1（向下移动）
        # 左: (-1, 0) → x减少1，y不变（向左移动）
        # 右: (1, 0)  → x增加1，y不变（向右移动）
        for i in range(self.nrow):
            for j in range(self.ncol):
                for a in range(4): # a表示4个change动作
                    if i==self.nrow-1 and j>0: #到达悬崖或终点
                        P[i*self.ncol+j][a]=[(1,i*self.ncol,0,True)]
                        continue

                    next_x=min(self.ncol-1,max(0,j+change[a][0]))
                    next_y=min(self.nrow-1,max(0,i+change[a][1]))
                    #计算下一个位置的x，y坐标

                    next_state=next_y*self.ncol+next_x #计算下一个位置的索引
                    reward=-1 # 每走一步有步数惩罚，实现最小步数
                    done=False #默认为未结束
                    if next_y==self.nrow-1 and next_x>0: #如果到最后一列，无论是悬崖还是终点都结束
                        done=True
                        if next_x!=self.ncol-1: #如果到悬崖，惩罚-100
                            reward=-100
                    P[i*self.ncol+j][a]=[(1,next_state,reward,done)] #把对应位置的索引和动作记录在状态转移方程P
        return P

In [36]:
# 4-3 policy_iteration 策略迭代算法
class PolicyIteration:
    def __init__(self,env,theta,gamma):
        self.env=env
        self.v=[0]*self.env.ncol*self.env.nrow # 初始化所有价值为0

        # 初始化为均匀分布的随机策略(4个动作均为0.25)
        self.pi=[[0.25,0.25,0.25,0.25] 
                 for i in range(self.env.ncol*self.env.nrow)]
        self.gamma=gamma # 折扣因子
        self.theta=theta # 策略函数收敛阈值
    def policy_evaluation(self):  # 策略评估
        cnt = 1  # 计数器
        while 1:
            max_diff = 0 # 最大价值变化
            new_v = [0] * self.env.ncol * self.env.nrow # 新价值函数
            for s in range(self.env.ncol * self.env.nrow):
                qsa_list = []  # 开始计算状态s下的所有Q(s,a)价值
                for a in range(4):
                    qsa = 0 
                    for res in self.env.P[s][a]:
                        p, next_state, r, done = res
                        qsa += p * (r + self.gamma * self.v[next_state] * (1 - done)) # 1-done确保终止状态无未来奖励
                        # 本章环境比较特殊,奖励和下一个状态有关,所以需要和状态转移概率相乘
                    qsa_list.append(qsa * self.pi[s][a]) # 动作加权概率归一化
                new_v[s] = sum(qsa_list)  # 状态价值函数和动作价值函数之间的关系——所有动作的价值期望求和
                max_diff = max(max_diff, abs(new_v[s] - self.v[s])) # 算所有位置价值函数的最大迭代差值
            self.v = new_v
            if max_diff < self.theta: break  # 所有位置的迭代都满足收敛条件,退出评估迭代
            cnt += 1
        print("策略评估进行%d轮后完成" % cnt)

    def policy_improvement(self):  # 策略提升
        for s in range(self.env.nrow * self.env.ncol): #对每一个位置遍历
            qsa_list = []
            for a in range(4): #遍历4个动作
                qsa = 0
                for res in self.env.P[s][a]:
                    p, next_state, r, done = res
                    qsa += p * (r + self.gamma * self.v[next_state] * (1 - done))
                qsa_list.append(qsa)
                
            maxq = max(qsa_list) # 找到序列里面的最大值
            cntq = qsa_list.count(maxq)  # 计算有几个动作得到了最大的Q值
            self.pi[s] = [1 / cntq if q == maxq else 0 for q in qsa_list] # 把概率权重分给四个动作里面的最大值
        print("策略提升完成")
        return self.pi

    def policy_iteration(self):  # 策略迭代
        while 1:
            self.policy_evaluation()
            old_pi = copy.deepcopy(self.pi)  # 将列表进行深拷贝,方便接下来进行比较
            new_pi = self.policy_improvement()
            if old_pi == new_pi: break

In [37]:
def print_agent(agent, action_meaning, disaster=[], end=[]):
    print("状态价值：")
    for i in range(agent.env.nrow):
        for j in range(agent.env.ncol):
            # 为了输出美观,保持输出6个字符
            print('%6.6s' % ('%.3f' % agent.v[i * agent.env.ncol + j]), end=' ')
        print()

    print("策略：")
    for i in range(agent.env.nrow):
        for j in range(agent.env.ncol):
            # 一些特殊的状态,例如悬崖漫步中的悬崖
            if (i * agent.env.ncol + j) in disaster:
                print('****', end=' ')
            elif (i * agent.env.ncol + j) in end:  # 目标状态
                print('EEEE', end=' ')
            else:
                a = agent.pi[i * agent.env.ncol + j]
                pi_str = ''
                for k in range(len(action_meaning)):
                    pi_str += action_meaning[k] if a[k] > 0 else 'o'
                print(pi_str, end=' ')
        print()


env = CliffWalkingEnv()
action_meaning = ['^', 'v', '<', '>']
theta = 0.001
gamma = 0.9
agent = PolicyIteration(env, theta, gamma)
agent.policy_iteration()
print_agent(agent, action_meaning, list(range(37, 47)), [47]) # 根据索引算出来的37-46悬崖，47终点

策略评估进行60轮后完成
策略提升完成
策略评估进行72轮后完成
策略提升完成
策略评估进行44轮后完成
策略提升完成
策略评估进行12轮后完成
策略提升完成
策略评估进行1轮后完成
策略提升完成
状态价值：
-7.712 -7.458 -7.176 -6.862 -6.513 -6.126 -5.695 -5.217 -4.686 -4.095 -3.439 -2.710 
-7.458 -7.176 -6.862 -6.513 -6.126 -5.695 -5.217 -4.686 -4.095 -3.439 -2.710 -1.900 
-7.176 -6.862 -6.513 -6.126 -5.695 -5.217 -4.686 -4.095 -3.439 -2.710 -1.900 -1.000 
-7.458  0.000  0.000  0.000  0.000  0.000  0.000  0.000  0.000  0.000  0.000  0.000 
策略：
ovo> ovo> ovo> ovo> ovo> ovo> ovo> ovo> ovo> ovo> ovo> ovoo 
ovo> ovo> ovo> ovo> ovo> ovo> ovo> ovo> ovo> ovo> ovo> ovoo 
ooo> ooo> ooo> ooo> ooo> ooo> ooo> ooo> ooo> ooo> ooo> ovoo 
^ooo **** **** **** **** **** **** **** **** **** **** EEEE 


In [38]:
 #  4-4 value_iteration 价值迭代算法
class ValueIteration:
    def __init__(self,env,theta,gamma):
        self.theta=theta
        self.gamma=gamma
        self.env=env
        self.v=[0]*(env.ncol*env.nrow)
        self.pi=[None for i in range(self.env.ncol*self.env.nrow)]
    def value_iteration(self):
        cnt=0
        while 1:
            max_diff=0
            new_v=[0]*self.env.ncol*self.env.nrow
            for s in range(self.env.ncol*self.env.nrow):
                qsa_list=[]
                for a in range(4):
                    qsa=0
                    for res in self.env.P[s][a]:
                        p,next_state,r,done=res
                        qsa+=p*(r+self.gamma*self.v[next_state]*(1-done))
                    qsa_list.append(qsa) # 相比策略评估没有概率加权，因为我们要的是价值最高的动作
                new_v[s]=max(qsa_list)
                max_diff=max(max_diff,abs(self.v[s]-new_v[s]))
            self.v=new_v
            if max_diff<self.theta:
                break
            cnt+=1
        print("价值迭代一共进行%d轮" %cnt)
        self.get_policy()

    def get_policy(self):
        for s in range(self.env.nrow*self.env.ncol):
            qsa_list=[]
            for a in range(4):
                qsa=0
                for res in self.env.P[s][a]:
                    p,next_state,r,done=res
                    qsa+=p*(r+self.gamma*(self.v[next_state])*(1-done))
                qsa_list.append(qsa)
            maxq=max(qsa_list)
            cntq=qsa_list.count(maxq)
            self.pi[s]=[1/cntq if q==maxq else 0 for q in qsa_list]



In [39]:
env = CliffWalkingEnv()
action_meaning = ['^', 'v', '<', '>']
theta = 0.001
gamma = 0.9
agent = ValueIteration(env, theta, gamma)
agent.value_iteration()
print_agent(agent, action_meaning, list(range(37, 47)), [47])

价值迭代一共进行14轮
状态价值：
-7.712 -7.458 -7.176 -6.862 -6.513 -6.126 -5.695 -5.217 -4.686 -4.095 -3.439 -2.710 
-7.458 -7.176 -6.862 -6.513 -6.126 -5.695 -5.217 -4.686 -4.095 -3.439 -2.710 -1.900 
-7.176 -6.862 -6.513 -6.126 -5.695 -5.217 -4.686 -4.095 -3.439 -2.710 -1.900 -1.000 
-7.458  0.000  0.000  0.000  0.000  0.000  0.000  0.000  0.000  0.000  0.000  0.000 
策略：
ovo> ovo> ovo> ovo> ovo> ovo> ovo> ovo> ovo> ovo> ovo> ovoo 
ovo> ovo> ovo> ovo> ovo> ovo> ovo> ovo> ovo> ovo> ovo> ovoo 
ooo> ooo> ooo> ooo> ooo> ooo> ooo> ooo> ooo> ooo> ooo> ovoo 
^ooo **** **** **** **** **** **** **** **** **** **** EEEE 


In [ ]:
# 4-5 冰湖环境代码
import gymnasium as gym
import pygame #引入可视化库
env = gym.make("FrozenLake-v1",render_mode="ansi") # 加上ansi参数之后才能显示 SFFF 这样的形式
env=env.unwrapped # 解封装才能访问状态转移矩阵
obs, info = env.reset()   # 在调用render()前，先初始化状态 self.s
print(env.render()) # 环境渲染

holes=set()
ends=set()
for s in env.P:
    for a in env.P[s]:
        for s_ in env.P[s][a]: # 这个s_是上面的res四元组(p,next_state,r,done)
            if s_[2] == 1.0:  # 获得奖励为1,代表是目标
                ends.add(s_[1])
            if s_[3] == True:
                holes.add(s_[1])
holes = holes - ends
print("冰洞的索引:", holes)
print("目标的索引:", ends)

for a in env.P[14]:  # 查看目标左边一格的状态转移信息
    print(env.P[14][a]) #滑动方向，除了身后，向其他三个方向(包括原地)各1/3概率

# 策略迭代
action_meaning = ['<', 'v', '>', '^']
theta = 1e-5
gamma = 0.9
agent = PolicyIteration(env, theta, gamma)
agent.policy_iteration()
print_agent(agent, action_meaning, [5, 7, 11, 12], [15])



SFFF
FHFH
FFFH
HFFG

冰洞的索引: {11, 12, 5, 7}
目标的索引: {15}
[(0.33333333333333337, 10, 0, False), (0.3333333333333333, 13, 0, False), (0.33333333333333337, 14, 0, False)]
[(0.33333333333333337, 13, 0, False), (0.3333333333333333, 14, 0, False), (0.33333333333333337, 15, 1, True)]
[(0.33333333333333337, 14, 0, False), (0.3333333333333333, 15, 1, True), (0.33333333333333337, 10, 0, False)]
[(0.33333333333333337, 15, 1, True), (0.3333333333333333, 10, 0, False), (0.33333333333333337, 13, 0, False)]
策略评估进行25轮后完成
策略提升完成
策略评估进行58轮后完成
策略提升完成
状态价值：
 0.069  0.061  0.074  0.056 
 0.092  0.000  0.112  0.000 
 0.145  0.247  0.300  0.000 
 0.000  0.380  0.639  0.000 
策略：
<ooo ooo^ <ooo ooo^ 
<ooo **** <o>o **** 
ooo^ ovoo <ooo **** 
**** oo>o ovoo EEEE 


In [41]:
# 价值迭代
action_meaning = ['<', 'v', '>', '^']
theta = 1e-5
gamma = 0.9
agent = ValueIteration(env, theta, gamma)
agent.value_iteration()
print_agent(agent, action_meaning, [5, 7, 11, 12], [15])

价值迭代一共进行60轮
状态价值：
 0.069  0.061  0.074  0.056 
 0.092  0.000  0.112  0.000 
 0.145  0.247  0.300  0.000 
 0.000  0.380  0.639  0.000 
策略：
<ooo ooo^ <ooo ooo^ 
<ooo **** <o>o **** 
ooo^ ovoo <ooo **** 
**** oo>o ovoo EEEE 
